In [1]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
# Uninstall old version to clear Kaggle cache without breaking Torchvision
!pip uninstall -y tiger
!pip install --no-cache-dir -e ".[dev,vlm,gen]" -q
# Confirm import-abo is registered
!python -m tiger.cli --help | grep import

Cloning into 'TIGeR-Text-Image-Generative-Repair'...
remote: Enumerating objects: 832, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 832 (delta 29), reused 40 (delta 21), pack-reused 779 (from 1)
Receiving objects: 100% (832/832), 3.41 MiB | 12.24 MiB/s, done.
Resolving deltas: 100% (345/345), done.
/kaggle/working/TIGeR-Text-Image-Generative-Repair
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tiger (pyproject.toml) ... done
             {synthgen,import-fashion,import-abo,noise,calibrate,detect,evaluate,analyze,train-arbiter,route,repair,sweep,calibrate-fusion,ablate,ablate-repair,compare-encoders,generate}
  {synthgen,import-fashion,import-abo,noise,calibrate,detect,evaluate,analyze,train-arbiter,route,repair,sweep,calibrate-fusion,

## 1. Import ABO Dataset
Make sure you have added the `khyeh0719/amazon-berkeley-objects-small` dataset to this notebook.

In [2]:
!python -m tiger.cli import-abo \
    --listings-dir /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-listings/listings/metadata \
    --images-csv /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/metadata/images.csv \
    --images-dir /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/small


Importing ABO from:
  listings : /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-listings/listings/metadata
  images   : /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/metadata/images.csv
  img dir  : /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/small
category     split      
electronics  calibration    2300
             report         2244
furniture    calibration     147
             report          193
home_decor   calibration      53
             report           63


### Validate Data Import
This ensures the notebook stops immediately if the ABO data wasn't found or parsed correctly.

In [3]:
import pandas as pd
from pathlib import Path

parquet_file = Path('data/sample/products.parquet')
assert parquet_file.exists(), "Data extraction failed: products.parquet not found! Check your --listings-dir and --images-dir paths."

df = pd.read_parquet(parquet_file)
print(f"Successfully imported {len(df)} products.")
assert len(df) > 0, "Data extraction failed: Parquet file is empty!"


Successfully imported 5000 products.


## 2. Calibrate on ABO
This fits the new similarity thresholds for the non-fashion domain.

In [4]:
!python -m tiger.cli calibrate

locked thresholds -> /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/thresholds/tiger_locked_thresholds.json
LOO calibration -> /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/thresholds/tiger_loo_calibration.json
verify epsilon=0.0374 (by category: {'electronics': 0.0369, 'furniture': 0.0284, 'home_decor': 0.064}) -> /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/thresholds/tiger_verify_calibration.json


## 3. Inject Synthetic Noise
This injects errors into the report split so there is something to repair. **This step is required** — without it, the Arbiter and ablation have no corrupted products to work on.

In [5]:
!python -m tiger.cli noise --seed 7

wrote /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/processed/noisy_report_seed7.parquet
{
  "seed": 7,
  "copies_per_row": 1,
  "rates": {
    "swap_image": 0.1,
    "swap_image_same_category": 0.03,
    "color_flip": 0.06,
    "near_color_flip": 0.02,
    "material_flip": 0.02,
    "attribute_drop": 0.02,
    "title_contradiction": 0.02,
    "mixed_swap_color": 0.02,
    "missing_image": 0.01
  },
  "rows_total": 2500,
  "rows_noisy": 750,
  "by_label": {
    "clean": 1750,
    "mutate_text": 350,
    "swap_image": 325,
    "mixed": 50,
    "missing_image": 25
  },
  "by_subtype": {
    "swap_image": 250,
    "color_flip": 150,
    "swap_image_same_category": 75,
    "attribute_drop": 50,
    "title_contradiction": 50,
    "near_color_flip": 50,
    "mixed_swap_color": 50,
    "material_flip": 50,
    "missing_image": 25
  },
  "self_verified": true
}


## 4. Retrain Arbiter
This retrains the Logistic Regression router on the new ABO-domain noise patterns.

In [6]:
!python -m tiger.cli train-arbiter

wrote /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/processed/noisy_report_cal_seed1007.parquet
{
  "seed": 1007,
  "copies_per_row": 1,
  "rates": {
    "swap_image": 0.1,
    "swap_image_same_category": 0.03,
    "color_flip": 0.06,
    "near_color_flip": 0.02,
    "material_flip": 0.02,
    "attribute_drop": 0.02,
    "title_contradiction": 0.02,
    "mixed_swap_color": 0.02,
    "missing_image": 0.01
  },
  "rows_total": 2500,
  "rows_noisy": 750,
  "by_label": {
    "clean": 1750,
    "mutate_text": 350,
    "swap_image": 325,
    "mixed": 50,
    "missing_image": 25
  },
  "by_subtype": {
    "swap_image": 250,
    "color_flip": 150,
    "swap_image_same_category": 75,
    "material_flip": 50,
    "attribute_drop": 50,
    "title_contradiction": 50,
    "near_color_flip": 50,
    "mixed_swap_color": 50,
    "missing_image": 25
  },
  "self_verified": true
}


📊 Stage 1: Error Detection Complete
--------------------------------------------------
Out of 2500 products scan

## 5. Run Repair Ablation
This evaluates the repair pipeline using the Independent Verifier (SigLIP) and Generative Fallback.

In [7]:
!python -m tiger.cli ablate-repair --independent --generative-fallback

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading pipeline components...: 100%|█████████████| 7/7 [00:02<00:00,  2.41it/s]
  Sampled 750 corrupted + 1500 clean = 2250 total rows

Running repair ablations on the full dataset...
1/5: Running 'Full System'...
[Generative Fallback] Synthesizing (SDXL-Turbo): 'Amazon Brand - Solimo Designer Girl Boss On Pink Sparkle UV Printed Soft Back Case Mobile Cover for Nokia 3.2. Category: electronics. Attributes: color=multicolour.'
[Generati

## 6b. Arbiter Confidence Diagnostic
Visualizes the Arbiter's `max(predict_proba())` distribution over all flagged ABO items.
This reveals whether the Gamma Gate (threshold=0.60) is ever firing and whether the Arbiter
is overconfident on out-of-domain data. See `ROADMAP_PROGRESS.md` H11 for context.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tiger.arbiter import ArbiterModel, featurize, CLASSES

# Load trained model
model_path = Path('data/thresholds/tiger_arbiter_model.json')
assert model_path.exists(), 'Run train-arbiter first'
model = ArbiterModel.from_json(model_path.read_text())

# Load one evidence file from the calibration run
ev_files = sorted(Path('data/outputs').glob('evidence_cal_seed*.jsonl'))
assert ev_files, 'No evidence JSONL files found — run train-arbiter first'
ev_file = ev_files[-1]  # use the last calibration seed

records = [json.loads(l) for l in ev_file.read_text().splitlines() if l.strip()]
print(f'Loaded {len(records)} evidence records from {ev_file.name}')

# Compute max confidence per record and predicted class
max_probs = []
pred_classes = []
for ev in records:
    p = model.predict_proba(featurize(ev))
    top = max(p, key=p.get)
    max_probs.append(p[top])
    pred_classes.append(top)

max_probs = np.array(max_probs)
GAMMA = 0.60

print(f'\nArbiter Confidence Stats (n={len(max_probs)})')
print(f'  Mean max-p  : {max_probs.mean():.3f}')
print(f'  Median max-p: {np.median(max_probs):.3f}')
print(f'  Min max-p   : {max_probs.min():.3f}')
print(f'  % below gamma={GAMMA}: {(max_probs < GAMMA).mean()*100:.1f}%')
print(f'  Predicted class distribution: { {c: pred_classes.count(c) for c in CLASSES} }')

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(max_probs, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(GAMMA, color='red', linestyle='--', linewidth=1.5, label=f'Gamma threshold ({GAMMA})')
ax.set_xlabel('Max Arbiter Confidence (max p over classes)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('ABO Arbiter Confidence Distribution\n'
             '(If most mass is right of red line, gamma gate never fires = overconfident Arbiter)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('data/outputs/arbiter_confidence_abo.png', dpi=150)
plt.show()
print('Saved: data/outputs/arbiter_confidence_abo.png')

## 6c. ABO Gamma Recalibration
The diagnostic above shows the Arbiter is overconfident on ABO — confidence scores are
clustered above the default gamma=0.60, so the gate never fires.
This cell computes a domain-specific gamma at the 25th percentile of the observed
confidence distribution, patches the config, and reruns the ablation.
The new table should show Full System > No Gamma Gate.

In [ ]:
import json, yaml, subprocess
import numpy as np
from pathlib import Path
from tiger.arbiter import ArbiterModel, featurize

# 1. Load model and compute confidence distribution
model = ArbiterModel.from_json(Path('data/thresholds/tiger_arbiter_model.json').read_text())
ev_files = sorted(Path('data/outputs').glob('evidence_cal_seed*.jsonl'))
records = [json.loads(l) for l in ev_files[-1].read_text().splitlines() if l.strip()]
max_probs = np.array([max(model.predict_proba(featurize(ev)).values()) for ev in records])

# 2. Set new gamma at 25th percentile (bottom 25% escalates)
new_gamma = float(np.percentile(max_probs, 25))
print(f'Default gamma : 0.60')
print(f'ABO gamma (p25): {new_gamma:.3f}')
print(f'Items that would now escalate via gamma: {(max_probs < new_gamma).sum()} / {len(max_probs)}')

# 3. Patch configs/tiger.yaml with new gamma
cfg_path = Path('configs/tiger.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
if 'arbiter' not in cfg:
    cfg['arbiter'] = {}
cfg['arbiter']['gamma'] = round(new_gamma, 3)
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False))
print(f'\nPatched configs/tiger.yaml: arbiter.gamma = {new_gamma:.3f}')

# 4. Rerun ablation with new gamma
print('\nRunning ablate-repair with recalibrated gamma...')
result = subprocess.run(
    ['python', '-m', 'tiger.cli', 'ablate-repair', '--independent', '--generative-fallback'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# 5. Show before/after comparison
import pandas as pd, glob
csvs = sorted(glob.glob('data/outputs/repair_ablations_summary*.csv'))
if csvs:
    df = pd.read_csv(csvs[-1])
    print('\n=== Ablation Results with Recalibrated Gamma ===')
    print(df.to_string())
    print('\nIf Full System > No Gamma Gate (gamma=0), gamma recalibration is working.')


## 6. View Results
Compare this table with the Fashion results in your paper.

In [8]:
import pandas as pd
import glob
# Dynamically find whichever ablation CSV was produced
csvs = sorted(glob.glob('data/outputs/repair_ablations_summary*.csv'))
print('Found CSVs:', csvs)
if csvs:
    df = pd.read_csv(csvs[-1])
    print(df.to_string())
else:
    print('No results CSV found. Check that ablate-repair ran successfully.')

Found CSVs: ['data/outputs/repair_ablations_summary.csv']
             Configuration  Repaired  Escalated  Total Attempted  Color Accuracy  V2T Cases
0      No Arbiter (Random)        40        439                0        0.000000          4
1             No VLM Judge        44        435                0        0.333333         12
2   No Generative Fallback        14        440                0        0.333333          9
3  No Gamma Gate (gamma=0)        39        440                0        0.333333          9
4              Full System        39        440                0        0.333333          9
